In [1]:
import pandas as pd
import numpy as np
from typing import Optional
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

import os
import time
import requests
from tqdm import tqdm

def get_yoy_growth_ranking(
    db_info: dict,
    item_name: str,
    table_name: str = "sec_financial_data",
    as_of_date: Optional[str] = None,
    top_n: int = 50,
) -> pd.DataFrame:

    user = db_info["user"]
    password = quote_plus(db_info["password"])
    host = db_info.get("host", "127.0.0.1")
    port = int(db_info.get("port", 3306))
    database = db_info["database"]

    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}?charset=utf8mb4"
    )

    where_date = ""
    params = {"item_name": item_name}
    if as_of_date:
        where_date = " AND date <= :as_of_date "
        params["as_of_date"] = as_of_date

    sql = f"""
        SELECT ticker, date, value
        FROM {table_name}
        WHERE item_name = :item_name
          AND value IS NOT NULL
          {where_date}
    """

    df = pd.read_sql(text(sql), engine, params=params)   # ✅ 핵심: text(sql)

    if df.empty:
        return pd.DataFrame(columns=["ticker","prev_value","curr_value","growth_rate","turn_flag","curr_date","prev_date"])

    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["ticker", "date", "value"])

    df = df.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")

    idx_latest = df.groupby("ticker")["date"].idxmax()
    curr = df.loc[idx_latest, ["ticker", "date", "value"]].rename(
        columns={"date": "curr_date", "value": "curr_value"}
    )

    curr["prev_date"] = curr["curr_date"] - pd.DateOffset(years=1)

    prev_lookup = df.rename(columns={"date": "prev_date", "value": "prev_value"})
    merged = curr.merge(prev_lookup[["ticker", "prev_date", "prev_value"]], on=["ticker", "prev_date"], how="left")
    merged = merged.dropna(subset=["prev_value"])

    if merged.empty:
        return pd.DataFrame(columns=["ticker","prev_value","curr_value","growth_rate","turn_flag","curr_date","prev_date"])

    prev = merged["prev_value"].astype(float)
    currv = merged["curr_value"].astype(float)

    merged["growth_rate"] = np.where(
        prev == 0,
        np.nan,
        (currv - prev) / np.abs(prev)
    )
    merged = merged.dropna(subset=["growth_rate"])
    merged["growth_rate"] = merged["growth_rate"] * 100.0

    merged["turn_flag"] = ""
    merged.loc[(merged["prev_value"] < 0) & (merged["curr_value"] > 0), "turn_flag"] = "turn_black"
    merged.loc[(merged["prev_value"] > 0) & (merged["curr_value"] < 0), "turn_flag"] = "turn_red"

    merged = merged.sort_values("growth_rate", ascending=False)

    out = merged[["ticker","prev_value","curr_value","growth_rate","turn_flag","curr_date","prev_date"]].head(top_n).reset_index(drop=True)
    return out

def get_json(url: str, session: requests.Session, timeout: int = 20):
    r = session.get(url, timeout=timeout)
    r.raise_for_status()
    return r.json()

def _fetch_with_backoff(session, url, max_retries=6, base_wait=1.0, timeout=20):
    """
    429/일시적 오류에 대해 지수 백오프 재시도
    """
    for attempt in range(max_retries + 1):
        try:
            r = session.get(url, timeout=timeout)
            if r.status_code == 429:
                wait = base_wait * (2 ** attempt)
                # Retry-After 헤더가 있으면 우선 적용
                ra = r.headers.get("Retry-After")
                if ra:
                    try:
                        wait = max(wait, float(ra))
                    except:
                        pass
                print(f"[RATE LIMIT] 429 -> sleep {wait:.1f}s (attempt {attempt}/{max_retries})")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except requests.RequestException as e:
            # 네트워크/일시적 오류도 백오프
            wait = base_wait * (2 ** attempt)
            print(f"[WARN] request error -> sleep {wait:.1f}s (attempt {attempt}/{max_retries}) | {e}")
            time.sleep(wait)

    # 끝까지 실패
    return None


def add_latest_price_mcap_from_fmp_quote_batch(
    ranking_df: pd.DataFrame,
    apikey: str,
    batch_size: int = 50,
    base_wait: float = 1.0,
) -> pd.DataFrame:
    """
    FMP quote를 배치로 호출해 price, marketCap을 가져와 ranking_df에 병합.
    - batch_size를 키우면 호출 횟수가 줄어 429가 크게 완화됩니다.
    """
    tickers = ranking_df["ticker"].dropna().astype(str).unique().tolist()

    rows = []
    with requests.Session() as sess:
        for i in tqdm(range(0, len(tickers), batch_size), desc="FMP quote batch"):
            batch = tickers[i:i + batch_size]
            tick_str = ",".join(batch)

            url = f"https://financialmodelingprep.com/api/v3/quote/{tick_str}?apikey={apikey}"
            js = _fetch_with_backoff(sess, url, base_wait=base_wait)

            if js is None:
                # 배치 전체 실패 시, 일단 None으로 채워두고 넘어감
                for t in batch:
                    rows.append({"ticker": t, "price": None, "marketCap": None})
                continue

            # js는 보통 list
            if isinstance(js, list):
                for q in js:
                    rows.append({
                        "ticker": q.get("symbol"),
                        "price": q.get("price"),
                        "marketCap": q.get("marketCap"),
                    })

            # 혹시 데이터 누락된 ticker가 있으면 None 채움
            got = set([r["ticker"] for r in rows if r.get("ticker")])
            for t in batch:
                if t not in got:
                    rows.append({"ticker": t, "price": None, "marketCap": None})

            # 배치 간 약간 쉬어주면 더 안정적
            time.sleep(0.3)

    fmp_df = pd.DataFrame(rows).drop_duplicates("ticker", keep="last")
    fmp_df["price"] = pd.to_numeric(fmp_df["price"], errors="coerce")
    fmp_df["marketCap"] = pd.to_numeric(fmp_df["marketCap"], errors="coerce")

    out = ranking_df.merge(fmp_df, on="ticker", how="left")
    return out

#
# # =========================
# # 사용 예시
# # =========================
# # apikey = os.environ.get("FMP_API_KEY")  # 권장
# apikey = "여기에_본인_APIKEY"  # (가능하면 환경변수 사용)
#
# ranking_df2 = add_latest_price_mcap_from_fmp_quote(
#     ranking_df=ranking_df,
#     apikey=apikey,
#     sleep_sec=0.12
# )
#
# print(ranking_df2.head(20))



In [2]:
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

# 예: 매출 item_name이 'revenue' 라고 가정
ranking_df = get_yoy_growth_ranking(
    db_info=db_info,
    item_name="revenue",          # <-- 여기만 바꾸면 됩니다 (예: "total_liabilities")
    table_name="sec_financial_data",
    as_of_date=None,              # 예: "2025-12-31" 넣으면 그 날짜 이하에서 최신값 기준
    top_n=400
)

print(ranking_df)


    ticker    prev_value    curr_value   growth_rate turn_flag  curr_date  \
0     JOBY  2.800000e+04  2.257400e+07  80521.428571           2025-09-30   
1     NIXX  1.358860e+05  3.191490e+07  23386.525470           2025-09-30   
2      QXO  1.310000e+07  2.728300e+09  20726.717557           2025-09-30   
3     HYPD  1.625000e+03  3.025060e+05  18515.753846           2025-09-30   
4     GXAI  2.704000e+03  4.982710e+05  18327.181953           2025-09-30   
..     ...           ...           ...           ...       ...        ...   
395   IMCR  8.024800e+07  1.036930e+08     29.215681           2025-09-30   
396    VRT  2.073500e+09  2.675800e+09     29.047504           2025-09-30   
397   GILT  1.527090e+08  1.970070e+08     29.008113           2025-06-30   
398   CODA  5.476540e+06  7.064800e+06     29.001158           2025-07-31   
399   SWAG  2.014400e+07  2.598100e+07     28.976370           2025-09-30   

     prev_date  
0   2024-09-30  
1   2024-09-30  
2   2024-09-30  
3   20

In [3]:
ranking_df

,ticker,prev_value,curr_value,growth_rate,turn_flag,curr_date,prev_date
0,JOBY,2.800000e+04,2.257400e+07,80521.428571,,2025-09-30,2024-09-30
1,NIXX,1.358860e+05,3.191490e+07,23386.525470,,2025-09-30,2024-09-30
2,QXO,1.310000e+07,2.728300e+09,20726.717557,,2025-09-30,2024-09-30
3,HYPD,1.625000e+03,3.025060e+05,18515.753846,,2025-09-30,2024-09-30
4,GXAI,2.704000e+03,4.982710e+05,18327.181953,,2025-09-30,2024-09-30
...,...,...,...,...,...,...,...
395,IMCR,8.024800e+07,1.036930e+08,29.215681,,2025-09-30,2024-09-30
396,VRT,2.073500e+09,2.675800e+09,29.047504,,2025-09-30,2024-09-30
397,GILT,1.527090e+08,1.970070e+08,29.008113,,2025-06-30,2024-06-30
398,CODA,5.476540e+06,7.064800e+06,29.001158,,2025-07-31,2024-07-31


In [4]:
apikey = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"  # (가능하면 환경변수 사용)

ranking_df2 = add_latest_price_mcap_from_fmp_quote_batch(
    ranking_df=ranking_df,
    apikey=apikey,
    batch_size=50,   # 25~100 사이에서 조절
    base_wait=1.0
)

print(ranking_df2.head(20))

FMP quote batch: 100%|██████████| 8/8 [00:04<00:00,  1.73it/s]

   ticker  prev_value    curr_value   growth_rate turn_flag  curr_date  \
0    JOBY     28000.0  2.257400e+07  80521.428571           2025-09-30   
1    NIXX    135886.0  3.191490e+07  23386.525470           2025-09-30   
2     QXO  13100000.0  2.728300e+09  20726.717557           2025-09-30   
3    HYPD      1625.0  3.025060e+05  18515.753846           2025-09-30   
4    GXAI      2704.0  4.982710e+05  18327.181953           2025-09-30   
5    PAVS     68454.0  1.241300e+07  18033.345020           2025-09-30   
6    GITS        13.0  1.838000e+03  14038.461538           2025-09-30   
7    IPSC    791000.0  1.091640e+08  13700.758534           2025-09-30   
8    EDIT     61000.0  7.543000e+06  12265.573770           2025-09-30   
9    SEPN    176000.0  2.149500e+07  12113.068182           2025-09-30   
10   BSLK      5000.0  3.700000e+05   7300.000000           2025-09-30   
11     VS      3848.0  1.993470e+05   5080.535343           2025-09-30   
12   BBIO   2732000.0  1.207000e+08   

In [7]:
import pandas as pd
import os

# 1. 경로 및 파일명 설정
path = r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis'
file_name = 'sales_growth_company_filtered_marketcap.xlsx'

# 2. 전체 저장 경로 생성
full_path = os.path.join(path, file_name)

# 3. 데이터프레임 저장 (index=False 옵션으로 인덱스 열 제외 가능)
# 질문하신 데이터프레임 변수명을 사용합니다.
sales_growth_company_filtered_marketcap.to_excel(full_path, index=False)

print(f"파일이 성공적으로 저장되었습니다: {full_path}")

파일이 성공적으로 저장되었습니다: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis\sales_growth_company_filtered_marketcap.xlsx
